In [ ]:
# Resolve project inputs consistently when Jupyter opens this subdirectory.
import os

from paths import ROOT

os.chdir(ROOT)
# Summarize correct model trials into the original condition/epoch features.
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch

data_dir = Path("artifacts/dpx/predictive/")

cue_probes = ["AX", "AY", "BX", "BY"]
dt = 25


def get_enc_df(recording):
    data_all = []
    for cp_idx, cue_probe in enumerate(sorted(recording.keys())):
        cue = cue_probe[0]
        probe = cue_probe[1]
        if cue_probe == "AX":
            response = "L"
        else:
            response = "R"
        hiddens_all = recording[cue_probe]["hiddens"]
        maxv = np.expand_dims(hiddens_all.max(axis=1), axis=1)
        hiddens_all = hiddens_all / maxv
        # minv = np.expand_dims(hiddens_all.max(axis=1), axis=1)
        # hiddens_all = (hiddens_all- minv) / (maxv- minv+1e-15)
        for trial in range(hiddens_all.shape[0]):
            corr = recording[cue_probe]["corrects"][trial]
            rt = recording[cue_probe]["RT"][trial]
            if not corr:
                continue
            hiddens = hiddens_all[trial, :]

            before_plain = hiddens[500 // dt : 1500 // dt].mean(axis=0).tolist()
            cue_plain = hiddens[500 // dt : 1500 // dt].mean(axis=0).tolist()
            delay_plain = hiddens[1500 // dt : 2500 // dt].mean(axis=0).tolist()
            probe_plain = hiddens[2500 // dt : 3000 // dt].mean(axis=0).tolist()

            # after_plain = hiddens[3000 // dt :].mean(axis=0).tolist()
            after_plain = hiddens[2500 // dt + rt // dt :].mean(axis=0).tolist()
            data_all.append(
                [cue, probe, response]
                + before_plain
                + cue_plain
                + delay_plain
                + probe_plain
                + after_plain
            )

    df_rec = pd.DataFrame(
        data_all,
        columns=[
            "cue",
            "probe",
            "response",
        ]
        + [f"before_{i}" for i in np.arange(hiddens.shape[-1])]
        + [f"cue_{i}" for i in np.arange(hiddens.shape[-1])]
        + [f"delay_{i}" for i in np.arange(hiddens.shape[-1])]
        + [f"probe_{i}" for i in np.arange(hiddens.shape[-1])]
        + [f"after_{i}" for i in np.arange(hiddens.shape[-1])],
    )
    return df_rec


data_dir.mkdir(parents=True, exist_ok=True)
Path("artifacts/dpx/predictive").mkdir(parents=True, exist_ok=True)

In [ ]:
# Load trusted local recordings and export reaction times for correct trials.
pred_all = True
model_family = "predictive" if pred_all else "classical"
recording_pre = torch.load(
    f"artifacts/dpx/{model_family}/rec_pre.pt",
    map_location="cpu",
    weights_only=False,
)
recording_bal = torch.load(
    f"artifacts/dpx/{model_family}/rec_bal.pt",
    map_location="cpu",
    weights_only=False,
)

recording_pre_classical = torch.load(
    "artifacts/dpx/classical/rec_pre.pt",
    map_location="cpu",
    weights_only=False,
)
recording_bal_classical = torch.load(
    "artifacts/dpx/classical/rec_bal.pt",
    map_location="cpu",
    weights_only=False,
)


def saveRT(recording, fn):
    RT_data = []
    for cp_idx, cue_probe in enumerate(sorted(recording.keys())):
        recording[cue_probe]["RT"]
        for corr, rt in zip(recording[cue_probe]["corrects"], recording[cue_probe]["RT"]):
            if corr:
                RT_data.append([cue_probe, int(rt)])
                # print(rt
        # RT_data += [[cue_probe, int(RT)] for RT in recording[cue_probe]["RT"]]

    df_RT = pd.DataFrame(RT_data, columns=["cue_probe", "RT"])
    df_RT.to_pickle(fn)
    return df_RT


df_RT_pre = saveRT(recording_pre, "artifacts/dpx/predictive/rt_pre.pkl")
df_RT_bal = saveRT(recording_bal, "artifacts/dpx/predictive/rt_bal.pkl")

saveRT(recording_pre_classical, "artifacts/dpx/classical/rt_pre.pkl")
saveRT(recording_bal_classical, "artifacts/dpx/classical/rt_bal.pkl")

In [ ]:
# Create balanced and prepotent response tables for both manuscript models.
# recording_pre_classical.keys()
df_rec_pre_classical = get_enc_df(recording_pre_classical)
# df_rec_pre_classical['trial_type'] = 'pre'
df_rec_bal_classical = get_enc_df(recording_bal_classical)
# df_rec_bal_classical['trial_type'] = 'bal'
df_rec_classical = pd.concat([df_rec_pre_classical, df_rec_bal_classical])


df_rec_pre = get_enc_df(recording_pre)
# df_rec_pre['trial_type'] = 'pre'
df_rec_bal = get_enc_df(recording_bal)
# df_rec_bal['trial_type'] = 'pre'
df_rec = pd.concat([df_rec_pre, df_rec_bal])

In [ ]:
# Save the feature tables consumed by the monkey/model matching notebook.
df_rec_pre_classical.to_pickle("artifacts/dpx/classical/table_pre.pkl")
df_rec_bal_classical.to_pickle("artifacts/dpx/classical/table_bal.pkl")

df_rec_pre.to_pickle("artifacts/dpx/predictive/table_pre.pkl")
df_rec_bal.to_pickle("artifacts/dpx/predictive/table_bal.pkl")

In [ ]:
# Average hidden activity within each DPX trial condition.
def extract_frs(recording):
    dat_hiddens_cube = []
    dat_hiddens_cube_all = []
    # cue_probes = sorted(recording.keys())
    print(cue_probes)

    for cp_idx, cue_probe in enumerate(cue_probes):
        dat_hiddens_cube.append(recording[cue_probe]["hiddens"].mean(axis=0))
        dat_hiddens_cube_all.append(recording[cue_probe]["hiddens"])
    print(recording[cue_probe].keys())
    dat_hiddens_cube = np.stack(dat_hiddens_cube)
    return dat_hiddens_cube, np.stack(dat_hiddens_cube_all)


fr_pre, fr_pre_all = extract_frs(recording_pre)
fr_bal, fr_bal_all = extract_frs(recording_bal)

In [ ]:
# Preserve the explicit model unit IDs and population summary used by the manuscript.
ns = np.array(
    [
        7,
        9,
        11,
        28,
        32,
        37,
        42,
        47,
        50,
        53,
        # 41,
        # 55,
        #
        #   new
        61,
        65,
        67,
        68,
        95,
        225,
        254,
    ]
)

pickle.dump(
    {
        "pre": {
            "AX": fr_pre[0],
            "AY": fr_pre[1],
            "BX": fr_pre[2],
            "BY": fr_pre[3],
        },
        "bal": {
            "AX": fr_bal[0],
            "AY": fr_bal[1],
            "BX": fr_bal[2],
            "BY": fr_bal[3],
        },
        "ids2plot": ns,
        "dt": dt,
    },
    open(data_dir / "population.pkl", "wb"),
)